<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 5 章：练习解答

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 5.1：使用启发式评分器打破自洽性方法中的平局

- 这一功能有多种实现方式
- 最简单的方式可能是在自洽性函数外部处理，直接使用它返回的字典（类似练习 4.4：我们把平局处理直接加入 `evaluate_math500_stream` 函数）
- 相关代码如下


```python
# ...
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # 已得到多数投票胜者
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### 新增：使用 heuristic_score 打破平局
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            # ...

    # ...
    return num_correct, num_examples, acc
```

- 下表展示了相对于第 3 章基线和第 4 章自洽性方法的改进

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 第 4 章的思维链提示基线    | 基础  | 33.4%    | 129.2 分钟 |
| 2 | 自洽性 (n=3) + 多数投票   | 基础  | 43.2%    | 328.2 分钟 |
| 3 | 自洽性 (n=3) + 启发式评分       | 基础  | 43.4%    | 326.5 分钟 |
| 4 | 自洽性 (n=3) + 平均对数概率    | 基础  | 44.8%    | 327.7 分钟 |

- 表中的准确率和运行时间是在 CUDA GPU（DGX Spark）上使用 MATH-500 测试集全部 500 个样本计算得到的

- 为方便起见，可以运行 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_consistency_scorer_math500.py) 脚本；该脚本位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

- 但需要注意，正如 [#159](https://github.com/rasbt/reasoning-from-scratch/issues/159) 中所讨论的，我们根据启发式分数确定多数票胜者时，只考虑每个多数票分组中最先出现的答案
- 例如：


&nbsp;
## 练习 5.2：在 Best-of-N 设置中使用启发式评分器

- Best-of-N 与自洽性方法类似，都会生成多个答案
- 不同之处在于，它不根据多数投票选择最终答案，而是使用评分函数（如 `heuristic_score`）对所有答案评分，并返回得分最高的答案
- 这一行为有多种实现方式；最简单的做法可能是以第 4 章现有的自洽性函数为模板，并按下面所示加入 `heuristic_score`


```python
# ...

from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- 结果如下

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 思维链提示基线 | 基础  | 33.4%    | 129.2 分钟 |
| 2 | Best-of-N (n=3) + 启发式评分              | 基础  | 40.6%    | 327.7 分钟 |
| 3 | Best-of-N (n=3) + 平均对数概率           | 基础  | 43.2%    | 330.2 分钟 |

- 表中的准确率和运行时间是在 CUDA GPU（DGX Spark）上使用 MATH-500 测试集全部 500 个样本计算得到的

- 为方便起见，可以运行 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本；该脚本位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## 练习 5.3：使用对数概率评分器打破自洽性方法中的平局

- 代码与练习 5.1 类似，只需将 `heuristic_score` 替换为 `avg_logprob_answer`。

```python
# ...
# from reasoning_from_scratch.ch05 import heuristic_score
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # 已得到多数投票胜者
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### 新增：使用 avg_logprob_answer 打破平局
            else:
                best = None
                best_score = float("-inf")
            
                # 考虑每个多数票分组中的所有成员
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            # ...

    # ...
    return num_correct, num_examples, acc
```

- 下表展示了相对于第 3 章基线和第 4 章自洽性方法的改进

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 思维链提示基线 | 基础  | 33.4%    | 129.2 分钟 |
| 2 | 自洽性 (n=3) + 多数投票   | 基础  | 43.2%    | 328.2 分钟 |
| 3 | 自洽性 (n=3) + 启发式评分       | 基础  | 43.4%    | 326.5 分钟 |
| 4 | 自洽性 (n=3) + 平均对数概率     | 基础  | 44.8%    | 327.7 分钟 |

- 表中的准确率和运行时间是在 CUDA GPU（DGX Spark）上使用 MATH-500 测试集全部 500 个样本计算得到的

- 为方便起见，可以运行 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本；该脚本位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## 练习 5.4：在 Best-of-N 设置中使用对数概率评分器

- 要使用对数概率评分器实现 Best-of-N，可以沿用练习 5.2 的代码，并将 `heuristic_score` 替换为 `avg_logprob_answer`：

```python

from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- 结果如下

| # | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 思维链提示基线 | 基础  | 33.4%    | 129.2 分钟 |
| 2 | Best-of-N (n=3) + 启发式评分              | 基础  | 待定      | 待定       |
| 3 | Best-of-N (n=3) + 平均对数概率           | 基础  | 待定      | 待定       |

- 表中的准确率和运行时间是在 CUDA GPU（DGX Spark）上使用 MATH-500 测试集全部 500 个样本计算得到的

- 为方便起见，可以运行 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本；该脚本位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)

&nbsp;
## 练习 5.5：使用启发式分数进行自我改进

- 使用 `heuristic_score` 实际上比使用对数概率分数更简单；只需修改以下代码：

```python
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)


torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 更新后的代码如下：

```python
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # 新增
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 使用启发式评分器的结果见第 4、5、10 行：

|    | 方法                 | 评分方式       | 迭代次数 | 模型      | 准确率 | 时间      |
|----|------------------------|---------------|------------|------------|----------|-----------|
| 1  | 基线（第 3 章）   | -             | -          | 基础       | 15.2%    | 10.1 分钟  |
| 2  | 自我改进        | 无          | 1          | 基础       | 25.0%    | 84.8 分钟  |
| 3  | 自我改进        | 无          | 2          | 基础       | 22.0%    | 165.4 分钟 |
| 4  | 自我改进        | 启发式评分     | 1          | 基础       | 21.6%    | 84.7 分钟  |
| 5  | 自我改进        | 启发式评分     | 2          | 基础       | 20.8%    | 151.4 分钟 |
| 6  | 自我改进        | 平均对数概率  | 1          | 基础       | 21.4%    | 85.3 分钟  |
| 7  | 自我改进        | 平均对数概率  | 2          | 基础       | 22.0%    | 165.3 分钟 |
|    |                        |               |            |            |          |           |
| 8  | 基线（第 3 章）   | -             | -          | 推理  | 48.2%    | 182.1 分钟 |
| 9  | 自我改进        | 无          | 1          | 推理  | 56.6%    | 498.8 分钟 |
| 10 | 自我改进        | 启发式评分     | 1          | 推理  | 57.8%    | 498.6 分钟 |
| 11 | 自我改进        | 平均对数概率  | 1          | 推理  | 48.4%    | 499.7 分钟 |

- 表中的准确率和运行时间是在 CUDA GPU（DGX Spark）上使用 MATH-500 测试集全部 500 个样本计算得到的
- 为方便起见，可以运行 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_refinement_math500.py) 脚本；该脚本位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)